In [14]:
import os
import glob
import json
import numpy as np
import matplotlib.pyplot as plt
root = "./data/fhr4/"
original_pos_dir = os.path.join(root, "final_data/pos")
augmented_pos_dir = os.path.join(root, "augmented_data/pos")
metadata_path = os.path.join(root, "final_data/dent_tracks_indices.json")
metadata_dict = json.load(open(metadata_path, "r"))

In [ ]:
pos_fns = glob.glob(original_pos_dir + "/*.npy")
for fn in pos_fns:
    pos_sample = np.load(fn)
    filename = fn.split('/')[-1].split('.')[0]
    track_idx = metadata_dict[filename]

    np.save(os.path.join(augmented_pos_dir, f"{filename}_track_{track_idx}.npy"), pos_sample)
    qc_output_dir = os.path.join(augmented_pos_dir, "qc")
    os.makedirs(qc_output_dir, exist_ok=True)
    plt.imshow(
        pos_sample.T, cmap="inferno", origin="lower", aspect="auto",
        interpolation="nearest",
    )
    plt.axis("off")
    plt.savefig(
        os.path.join(qc_output_dir, f"{filename}_track_{track_idx}.png"),
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.close()


    


        
    # Create vertical roll augmentations (small shifts for spectrograms)
    for i, shift_v in enumerate([2, 4, 6, 8, 10, 12, 14, 16, 18]):
        rolled_v = np.roll(pos_sample, shift_v, axis=0)  # Roll vertically
            # Create horizontal roll augmentations
        for j, shift_h in enumerate([50, 85, -50, -85]):
            rolled_vh = np.roll(rolled_v, shift_h, axis=1)  # Roll horizontally

            aug_filename = f"{filename}_track_{(track_idx+shift_v)%20}_hroll_{shift_h}.npy"
            np.save(os.path.join(augmented_pos_dir, aug_filename), rolled_vh)
            plt.imshow(
                rolled_vh.T, cmap="inferno", origin="lower", aspect="auto",
                interpolation="nearest",
            )
            plt.axis("off")
            plt.savefig(
                os.path.join(qc_output_dir, f"{aug_filename.split('.')[0]}.png"),
                bbox_inches="tight",
                pad_inches=0,
            )
            plt.close()
    